# 04 - Model Building
Feature selection, train 4 models using sklearn's built-in functions, cross-validate, and save models.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score)
import joblib

try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None
    print('xgboost not installed, skipping')

try:
    from lightgbm import LGBMClassifier
except ImportError:
    LGBMClassifier = None
    print('lightgbm not installed, skipping')

from config import (PROCESSED_DATA_DIR, MODELS_DIR, EXCLUDE_FROM_MODELING,
                    DATE_COLS, RANDOM_STATE, TEST_SIZE)
from src.util import load_dataframe, save_model

## 4.1 Load engineered data

In [ ]:
df = load_dataframe(os.path.join(PROCESSED_DATA_DIR, 'df_engineered.csv'))
print(f'Shape: {df.shape}')
print(f'Churn rate: {df["is_churned"].mean():.1%}')

## 4.2 Feature selection

In [ ]:
# get valid numeric feature columns (exclude IDs, raw text, dates, target)
date_parsed_cols = [c for c in df.columns if '_parsed' in c]
raw_date_cols = DATE_COLS + ['Registered Time']
exclude = set(EXCLUDE_FROM_MODELING + date_parsed_cols + raw_date_cols)

feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                if c not in exclude and c != 'is_churned']

print(f'Total candidate features: {len(feature_cols)}')
print(f'Excluded: {len(exclude)} columns')

In [ ]:
# rank features by Random Forest importance
X_all = df[feature_cols].copy()
y = df['is_churned'].copy()

# fill any remaining NaN/inf in features
X_all = X_all.fillna(0).replace([np.inf, -np.inf], 0)

rf_selector = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1, max_depth=10)
rf_selector.fit(X_all, y)

importance = pd.DataFrame({
    'Feature': X_all.columns,
    'Importance': rf_selector.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print('Top 15 features:')
for i, row in importance.head(15).iterrows():
    print(f'  {i+1:2d}. {row["Feature"]:40s} {row["Importance"]:.4f}')

In [ ]:
# plot feature importance (top 20)
fig, ax = plt.subplots(figsize=(10, 8))
top20 = importance.head(20)
ax.barh(range(len(top20)), top20['Importance'], color='steelblue', edgecolor='black')
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20['Feature'])
ax.invert_yaxis()
ax.set_xlabel('Importance')
ax.set_title('Top 20 Features by Random Forest Importance')
plt.tight_layout()
plt.show()

In [ ]:
# select features above median importance
threshold = importance['Importance'].median()
selected = importance[importance['Importance'] >= threshold]['Feature'].tolist()
print(f'Selected {len(selected)} features (importance >= {threshold:.4f})')

# remove multicollinear features (correlation > 0.85)
corr_matrix = X_all[selected].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop = set()
for col in upper.columns:
    for idx in upper.index:
        if upper.loc[idx, col] > 0.85:
            imp1 = importance[importance['Feature'] == idx]['Importance'].values
            imp2 = importance[importance['Feature'] == col]['Importance'].values
            imp1 = imp1[0] if len(imp1) > 0 else 0
            imp2 = imp2[0] if len(imp2) > 0 else 0
            drop_col = col if imp1 >= imp2 else idx
            to_drop.add(drop_col)
            print(f'  Correlated: {idx} <-> {col} (r={upper.loc[idx, col]:.2f}) -> drop {drop_col}')

if to_drop:
    selected = [f for f in selected if f not in to_drop]
    print(f'\nDropped {len(to_drop)} correlated features')

print(f'Final feature count: {len(selected)}')
print(f'Features: {selected}')

## 4.3 Train/test split

In [ ]:
# prepare features and target
X = df[selected].copy().fillna(0).replace([np.inf, -np.inf], 0)
y = df['is_churned'].copy()

# split using sklearn's train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# scale for logistic regression
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=selected, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=selected, index=X_test.index)

print(f'Train: {X_train.shape[0]:,} rows (churn rate: {y_train.mean():.1%})')
print(f'Test:  {X_test.shape[0]:,} rows (churn rate: {y_test.mean():.1%})')

## 4.4 Train models

In [ ]:
# compute class weight ratio for XGBoost / LightGBM
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight: {scale_pos:.2f}')

# define models
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=15, min_samples_split=10,
        random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced'
    ),
}

if XGBClassifier is not None:
    models['XGBoost'] = XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        random_state=RANDOM_STATE, n_jobs=-1, eval_metric='logloss',
        scale_pos_weight=scale_pos
    )

if LGBMClassifier is not None:
    models['LightGBM'] = LGBMClassifier(
        n_estimators=200, max_depth=8, learning_rate=0.1,
        random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
        scale_pos_weight=scale_pos
    )

print(f'Training {len(models)} models: {list(models.keys())}')

In [ ]:
# train each model and compute test metrics
results = {}

for name, model in models.items():
    print(f'\nTraining: {name}...')
    
    # use scaled data for logistic regression
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
    
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_proba),
        'AUC-PR': average_precision_score(y_test, y_proba),
    }
    
    print(f'  Accuracy:  {results[name]["Accuracy"]:.4f}')
    print(f'  Precision: {results[name]["Precision"]:.4f}')
    print(f'  Recall:    {results[name]["Recall"]:.4f}')
    print(f'  F1-Score:  {results[name]["F1-Score"]:.4f}')
    print(f'  AUC-ROC:   {results[name]["AUC-ROC"]:.4f}')
    print(f'  AUC-PR:    {results[name]["AUC-PR"]:.4f}')

## 4.5 Model comparison

In [ ]:
# comparison table
comparison = pd.DataFrame(results).T.round(4)
print(comparison.to_string())

# highlight best
best_model = comparison['F1-Score'].idxmax()
print(f'\nBest model (F1-Score): {best_model}')

## 4.6 Cross-validation

In [ ]:
# 5-fold stratified cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_results = {}
for name, model in models.items():
    if name == 'Logistic Regression':
        scores = cross_val_score(model, X_train_scaled, y_train, cv=skf, scoring='f1')
    else:
        scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='f1')
    
    cv_results[name] = {'Mean F1': scores.mean(), 'Std F1': scores.std()}
    print(f'{name:25s} F1: {scores.mean():.4f} +/- {scores.std():.4f}  ({scores})')

cv_df = pd.DataFrame(cv_results).T.round(4)
print('\n', cv_df.to_string())

## 4.7 Save models and artifacts

In [ ]:
# save trained models to models/ folder
os.makedirs(MODELS_DIR, exist_ok=True)

for name, model in models.items():
    safe_name = name.lower().replace(' ', '_')
    model_path = os.path.join(MODELS_DIR, f'{safe_name}.pkl')
    save_model(model, model_path, name)

# save training artifacts (features list, scaler, results)
artifacts = {
    'selected_features': selected,
    'scaler': scaler,
    'results': results,
    'cv_results': cv_results,
    'importance': importance,
    'best_model_name': best_model,
}
artifacts_path = os.path.join(PROCESSED_DATA_DIR, 'training_artifacts.pkl')
joblib.dump(artifacts, artifacts_path)
print(f'\nTraining artifacts saved to: {artifacts_path}')